# Evaluation and results

Evaluate the checkpoints from [diffusion and flow matching](diffusion_and_flow_matching.ipynb) and [deterministic ensembles](deterministic_ensembles.ipynb), then compare their saved results in Python. No earlier notebook is required: setup reuses existing runs or prepares small examples. If evaluation CSVs already exist, skip setup and evaluation.

| View | Question |
| --- | --- |
| Test metrics | How well does one forecast window predict from true inputs? |
| Rollout metrics | How does error grow when predictions are fed back? |
| CRPS, spread–skill and coverage | Is the predictive distribution useful and calibrated? |

## Prepare example runs (optional)

On a fresh start, run the tiny training tutorials in separate kernels (roughly a minute on a local CPU). Complete existing runs are reused, including the saved backbone choice. For your own experiments, skip this cell and edit the run selection below.

In [ ]:
from _support import prepare_tutorial

for tutorial in (
    "autoencoder_and_latents",
    "diffusion_and_flow_matching",
    "deterministic_ensembles",
):
    prepare_tutorial(tutorial)

## Select the trained runs

Reuse the backbone selection saved by the generative tutorial. Each model sees the same seeded test trajectories, with two input frames and two predicted frames per step. Processor-only checkpoints also need the saved autoencoder.

In [ ]:
import json

from _support import EVALUATION_OPTIONS, OUTPUT_ROOT, run_autocast

autoencoder_checkpoint = OUTPUT_ROOT / "autoencoder" / "autoencoder.ckpt"
selected_runs = json.loads((OUTPUT_ROOT / "processor_runs.json").read_text())
checkpoints = {
    name: OUTPUT_ROOT / path / "processor.ckpt" for name, path in selected_runs.items()
}
checkpoints["Deterministic ensemble"] = (
    OUTPUT_ROOT / "deterministic_ensemble" / "encoder_processor_decoder.ckpt"
)
eval_dirs = {
    name: checkpoint.parent / "eval" for name, checkpoint in checkpoints.items()
}
n_members = 5
max_rollout_steps = 4

## Evaluate once

Use the CLI to evaluate saved checkpoints in physical space. This CPU check uses two test batches and two rollout trajectories; remove the limits in [the shared settings](./_support.py) for a full evaluation. Rerun this cell when checkpoints or evaluation settings change.

In [ ]:
evaluation_options = [
    *EVALUATION_OPTIONS,
    "eval.mode=ambient",
    "eval.metrics=[rmse,crps,spread,skill,ssr]",
    "eval.batch_indices=[]",
    f"eval.n_members={n_members}",
    f"eval.max_rollout_steps={max_rollout_steps}",
    "eval.metric_windows=[null]",
    "eval.metric_windows_rollout=[null]",
    "eval.compute_rollout_coverage=true",
    "eval.compute_rollout_metrics=true",
]

for name, checkpoint in checkpoints.items():
    autoencoder_options = (
        [f"++autoencoder_checkpoint={autoencoder_checkpoint}"]
        if name in selected_runs
        else []
    )
    run_autocast(
        "eval",
        "--workdir",
        checkpoint.parent,
        *evaluation_options,
        *autoencoder_options,
    )

## Collate saved results

`RunCollator` joins training configurations and aggregate metrics. `config_params` selects configuration fields using dotted paths or wildcards; `N/A` means a setting is absent. It works with CLI runs too.

Keep only the selected runs, so older experiments cannot enter the comparison.

In [ ]:
import pandas as pd

from autocast.scripts.utils import RunCollator

config_columns = {
    "processor": "model.processor._target_",
    "learning_rate": "optimizer.learning_rate",
    "train_batch_size": "datamodule.batch_size",
    "sampling_steps": "model.processor.*_steps",
}
runs = RunCollator(OUTPUT_ROOT, config_params=config_columns).collate(save_csv=False)
run_labels = {
    str(checkpoint.parent.relative_to(OUTPUT_ROOT)): name
    for name, checkpoint in checkpoints.items()
}
runs = runs.set_index("run_path", drop=False).loc[list(run_labels)]
runs = runs.rename(index=run_labels).rename_axis("Run")
runs[list(config_columns)]

## Compare aggregate scores

Edit `compare` to select runs for the table and plots; these cells do not rerun evaluation. `overall_*` columns contain test scores, while `*_all` columns summarize the full evaluated rollout.

Lower RMSE and CRPS are better. A spread–skill ratio near one indicates that ensemble spread matches typical error magnitude.

In [ ]:
compare = list(checkpoints)  # or ["Flow matching", "Diffusion"]
score_columns = {
    "overall_rmse": "Test RMSE",
    "overall_crps": "Test CRPS",
    "rmse_all": "Rollout RMSE",
    "crps_all": "Rollout CRPS",
    "ssr_all": "Rollout SSR",
}
scores = runs.loc[compare, list(score_columns)].rename(columns=score_columns)
scores.style.format(precision=3)

## Inspect lead time

Load the detailed CSVs once. Change `metrics` below to explore forecast error or uncertainty, using the same selected runs.

In [ ]:
lead_time = {
    name: pd.read_csv(
        path / "rollout_metrics_per_timestep_channel_all.csv", index_col=0
    ).T.rename(index=int)
    for name, path in eval_dirs.items()
}
coverage = {
    name: pd.read_csv(path / "rollout_coverage_window_all.csv")
    for name, path in eval_dirs.items()
}

In [ ]:
import matplotlib.pyplot as plt

metrics = ("rmse", "crps")  # also "spread", "skill", "ssr"
fig, axes = plt.subplots(
    1, len(metrics), figsize=(5 * len(metrics), 3.5), squeeze=False
)
for axis, metric in zip(axes.flat, metrics, strict=True):
    for name in compare:
        values = lead_time[name][metric]
        axis.plot(values.index, values, marker="o", label=name)
    axis.set(xlabel="Lead-time index", ylabel=metric.upper())
    axis.grid(alpha=0.25)
axes.flat[-1].legend(frameon=False)
fig.tight_layout()
plt.show()

## Check coverage

A calibrated ensemble follows the diagonal: an 80% predictive interval should contain the truth about 80% of the time. These small ensembles and short trajectories only illustrate the diagnostic.

In [ ]:
fig, axis = plt.subplots(figsize=(5, 4))
for name in compare:
    values = coverage[name]
    axis.plot(
        values["coverage_level"],
        values["observed_mean"],
        marker="o",
        label=name,
    )
axis.plot([0, 1], [0, 1], "--", color="black", label="Ideal")
axis.set(
    xlabel="Nominal coverage", ylabel="Observed coverage", xlim=(0, 1), ylim=(0, 1)
)
axis.grid(alpha=0.25)
axis.legend(frameon=False)
fig.tight_layout()
plt.show()

## Export the comparison

Keep configuration fields, source run paths and metrics together for further analysis. For a real comparison, increase data and evaluation limits, match training budgets, repeat seeds and report uncertainty across runs.

In [ ]:
runs.loc[compare].to_csv(OUTPUT_ROOT / "collated_results.csv")